# Intrusão salina 1D — domínio suficientemente grande

Este notebook determina numericamente um comprimento de domínio suficiente para o caso sintético crítico $Q=2\,\mathrm{m^3/s}$, mantendo a condição de Danckwerts, $\Delta x=100\,\mathrm{m}$ e $\Delta t=60\,\mathrm{s}$.

Em domínios longos, a marcha ciclo a ciclo pode confundir um transiente muito lento com periodicidade. Por isso, esta versão resolve diretamente o ponto fixo do mapa de uma maré,

$$
F(C_0)=C_0,
$$

por GMRES matricialmente livre. O resíduo $\|F(C_0)-C_0\|_\infty$ verifica a periodicidade sem depender da condição inicial.

A coordenada $x=0$ representa uma seção efetiva situada a 10 km da foz. A captação a 41 km da foz corresponde a $x_{\mathrm{cap}}=31\,\mathrm{km}$. Os resultados continuam sendo sintéticos e não constituem previsão para o Rio São Mateus.


In [ ]:
# Execute esta célula uma vez, em uma sessão nova.
from pathlib import Path
import subprocess
import sys
import zipfile

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    archive = Path("/content/salt_intrusion_1d_v1.0.0.zip")
    if not archive.exists():
        raise FileNotFoundError(
            "Envie salt_intrusion_1d_v1.0.0.zip para /content e execute novamente."
        )
    with zipfile.ZipFile(archive) as compressed:
        compressed.extractall("/content")
    project_dir = Path("/content/salt_intrusion_1d")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--no-deps", "--force-reinstall", str(project_dir)]
    )
else:
    project_dir = Path("..").resolve()
    sys.path.insert(0, str(project_dir / "src"))
%matplotlib inline


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from salt_intrusion_1d.domain_sensitivity import (
    adjacent_domain_metrics,
    domain_metrics,
    plot_domain_convergence,
    plot_domain_sensitivity,
    run_domain_sensitivity,
    write_capture_timeseries,
    write_domain_sensitivity_summary,
)


## Solução periódica direta em domínios sucessivos

A malha permanece fixa em 100 m. Os comprimentos são adensados na região em que a frente se aproxima da estabilização. A execução pode levar alguns minutos no Colab, principalmente para $L\geq500\,\mathrm{km}$.


In [ ]:
lengths_km = (
    50.0, 70.0, 100.0, 150.0, 200.0, 250.0, 300.0,
    350.0, 400.0, 450.0, 500.0, 550.0, 600.0,
)

domain_results = run_domain_sensitivity(
    lengths_km=lengths_km,
    discharge_m3_s=2.0,
    dx_m=100.0,
    dt_s=60.0,
    solver="fixed_point",
)


## Resultados por domínio

O número de avaliações do mapa informa o custo do GMRES. A periodicidade é verificada diretamente pelo resíduo do ponto fixo, e não pela pequena variação entre ciclos consecutivos.


In [ ]:
for length_km, result in sorted(domain_results.items()):
    metrics = domain_metrics(result)
    status = "atingido" if metrics.converged else "não atingido"
    print(f"L = {length_km:.0f} km")
    print(f"  Ponto fixo periódico: {status}")
    print(f"  Avaliações do mapa de uma maré: {metrics.cycle_map_evaluations}")
    print(f"  Resíduo periódico: {metrics.periodic_residual_linf_psu:.3e} PSU")
    print(f"  Intrusão média: {metrics.mean_intrusion_from_mouth_km:.3f} km da foz")
    print(f"  Intrusão máxima: {metrics.max_intrusion_from_mouth_km:.3f} km da foz")
    print(f"  Distância da frente máxima à fronteira: {metrics.distance_front_to_boundary_km:.3f} km")
    print(f"  Salinidade média na captação: {metrics.capture_mean_salinity_psu:.3f} PSU")
    print(f"  Salinidade máxima na captação: {metrics.capture_max_salinity_psu:.3f} PSU")
    print(f"  Tempo acima de 0,5 PSU: {metrics.capture_time_above_threshold_h:.3f} h ({100 * metrics.capture_fraction_above_threshold:.1f}% do ciclo)")
    print()


## Convergência em relação ao comprimento


In [ ]:
fig = plot_domain_convergence(domain_results)
plt.show()


## Diferenças entre domínios consecutivos

Adotam-se dois critérios de suficiência:

$$
|\overline C_{\mathrm{cap}}^{\,L_2}-\overline C_{\mathrm{cap}}^{\,L_1}|<0{,}05\ \mathrm{PSU},
$$

e

$$
|\overline L_s^{\,L_2}-\overline L_s^{\,L_1}|<0{,}1\ \mathrm{km},
\qquad
|L_{s,\max}^{L_2}-L_{s,\max}^{L_1}|<0{,}1\ \mathrm{km}.
$$

Para evitar uma conclusão baseada em um único incremento favorável, exige-se confirmação em dois pares consecutivos.


In [ ]:
for comparison in adjacent_domain_metrics(domain_results):
    short = comparison["short_domain_km"]
    long = comparison["long_domain_km"]
    print(f"L = {short:g} km versus L = {long:g} km")
    print(f"  Diferença máxima na salinidade da captação: {comparison['capture_salinity_linf_psu']:.6f} PSU")
    print(f"  Diferença das médias na captação: {comparison['capture_mean_salinity_difference_psu']:.6f} PSU")
    print(f"  Diferença na intrusão média: {comparison['mean_intrusion_difference_m'] / 1_000:.6f} km")
    print(f"  Diferença na intrusão máxima: {comparison['max_intrusion_difference_m'] / 1_000:.6f} km")
    print()


## Exportação

Esta célula salva as tabelas, as séries na captação e os gráficos de convergência e sensibilidade.


In [ ]:
output_dir = Path("/content/results_domain_sensitivity_Q2_v0.6") if IN_COLAB else Path("../results_domain_sensitivity_Q2_v0.6")
output_dir.mkdir(parents=True, exist_ok=True)

write_domain_sensitivity_summary(domain_results, output_dir)
write_capture_timeseries(domain_results, output_dir)
plot_domain_convergence(
    domain_results,
    output_dir / "domain_convergence.png",
)
plot_domain_sensitivity(
    domain_results,
    output_dir / "domain_sensitivity.png",
)
plt.close("all")

print(f"Arquivos salvos em: {output_dir.resolve()}")


## Interpretação

A salinidade na captação estabiliza antes da posição da frente. A frente satisfaz o limite de 0,1 km nos pares 500--550 km e 550--600 km; portanto, $L=550\,\mathrm{km}$ é numericamente suficiente segundo os critérios adotados, e $L=600\,\mathrm{km}$ fornece a confirmação.

Esse resultado não torna $L=550\,\mathrm{km}$ uma escolha fisicamente adequada para o RSM. Pelo contrário: uma fronteira situada a 560 km da foz exigiria estender por centenas de quilômetros as hipóteses de seção uniforme, maré prescrita e dispersão constante no espaço. A necessidade de um domínio tão longo mostra que a parametrização sintética — sobretudo a relação entre a pequena velocidade fluvial e a dispersão efetiva — não deve ser usada para inferências quantitativas sobre o rio sem calibração.


## Resultado obtido nesta versão

| $L$ (km) | Intrusão média (km da foz) | Intrusão máxima (km da foz) | Média na captação (PSU) |
|---:|---:|---:|---:|
| 50 | 56,663 | 59,458 | 3,932 |
| 100 | 99,760 | 102,247 | 6,835 |
| 200 | 158,535 | 161,022 | 7,761 |
| 300 | 176,892 | 179,378 | 7,872 |
| 400 | 180,056 | 182,543 | 7,886 |
| 500 | 180,511 | 182,997 | 7,889 |
| 550 | 180,557 | 183,044 | 7,889 |
| 600 | 180,574 | 183,061 | 7,889 |

Nos pares finais:

- 500--550 km: diferenças de 0,000209 PSU na média da captação e 0,046 km na intrusão média e máxima;
- 550--600 km: diferenças de 0,000078 PSU na média da captação e 0,017 km na intrusão média e máxima.

Logo, o teste numérico identifica $L=550\,\mathrm{km}$ como suficiente, mas simultaneamente demonstra que esse comprimento é fisicamente impraticável para o modelo homogêneo do Rio São Mateus.
